<a href="https://colab.research.google.com/github/AshokGit544/Enterprise-Finance-KnowledgeOps-Copilot/blob/main/Enterprise_Finance_KnowledgeOps_Copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install pandas numpy pyspark sentence-transformers faiss-cpu langchain langchain-community transformers accelerate bitsandbytes

In [ ]:
!pip -q install requests==2.32.4

In [ ]:
import os
import json
import random
import zipfile
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

In [ ]:
random.seed(42)
np.random.seed(42)

PROJECT_DIR = Path("Enterprise-Finance-KnowledgeOps-Copilot")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
VECTOR_DIR = PROJECT_DIR / "data" / "vector_store"
OUTPUT_DIR = PROJECT_DIR / "data" / "output"
DAG_DIR = PROJECT_DIR / "dags"
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
CONFIG_DIR = PROJECT_DIR / "config"

for folder in [RAW_DIR, PROCESSED_DIR, VECTOR_DIR, OUTPUT_DIR, DAG_DIR, NOTEBOOK_DIR, CONFIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders created successfully.")
print("Project root:", PROJECT_DIR.resolve())

In [ ]:
vendors = pd.DataFrame([
    ["V001", "Alpha Energy Services", "Maintenance", "NY", "Active"],
    ["V002", "BlueGrid Technologies", "IT Services", "CT", "Active"],
    ["V003", "Northwind Electric", "Electrical Components", "MA", "Active"],
    ["V004", "GreenVolt Contractors", "Field Services", "PA", "Active"],
    ["V005", "Prime Utility Logistics", "Logistics", "NJ", "Review"],
    ["V006", "CoreAxis Industrial Supply", "Industrial Supply", "MI", "Active"]
], columns=["vendor_id", "vendor_name", "vendor_category", "vendor_state", "vendor_status"])


gl_accounts = pd.DataFrame([
    ["400100", "Revenue - Energy Delivery", "Revenue"],
    ["500100", "Opex - Maintenance", "Expense"],
    ["500200", "Opex - IT Services", "Expense"],
    ["500300", "Opex - Contractors", "Expense"],
    ["500400", "Opex - Logistics", "Expense"],
    ["500500", "Opex - Field Operations", "Expense"],
    ["110100", "Accounts Payable", "Liability"]
], columns=["gl_account", "gl_account_name", "gl_type"])


cost_centers = pd.DataFrame([
    ["CC100", "Transmission Operations", "Operations"],
    ["CC110", "Distribution Maintenance", "Operations"],
    ["CC120", "Substation Engineering", "Engineering"],
    ["CC130", "Customer Support", "Customer Ops"],
    ["CC140", "Finance Operations", "Finance"],
    ["CC150", "Grid Modernization", "Engineering"]
], columns=["cost_center_id", "cost_center_name", "department"])


finance_policies = pd.DataFrame([
    ["POL001", "Invoice Policy", "Invoices above 40000 USD require finance manager approval before payment release.", "High"],
    ["POL002", "Vendor Compliance Policy", "Invoices from vendors in review status must be flagged for manual compliance verification.", "High"],
    ["POL003", "Duplicate Control Policy", "Invoices with same vendor, amount, and close invoice date must be checked for duplicate risk.", "Medium"],
    ["POL004", "Missing Data Policy", "Invoices with missing vendor, missing description, or invalid amount must be blocked from downstream approval workflows.", "High"],
    ["POL005", "Late Payment Policy", "Payments delayed beyond 30 days from posting date should be investigated for operational or financial control gaps.", "Medium"]
], columns=["policy_id", "policy_name", "policy_text", "severity"])


vendors.to_csv(RAW_DIR / "vendors.csv", index=False)
gl_accounts.to_csv(RAW_DIR / "gl_accounts.csv", index=False)
cost_centers.to_csv(RAW_DIR / "cost_centers.csv", index=False)
finance_policies.to_csv(RAW_DIR / "finance_policies.csv", index=False)

print("Master data created successfully.")
print("Files saved:")
print("-", RAW_DIR / "vendors.csv")
print("-", RAW_DIR / "gl_accounts.csv")
print("-", RAW_DIR / "cost_centers.csv")
print("-", RAW_DIR / "finance_policies.csv")

In [ ]:
invoice_rows = []
payment_rows = []

start_date = datetime(2024, 1, 1)

descriptions = [
    "transformer repair service",
    "sap fico support work",
    "field contractor invoice",
    "substation maintenance work",
    "it platform support",
    "logistics support for equipment",
    "distribution network inspection",
    "grid modernization consulting",
    "vendor maintenance support"
]

for i in range(1, 151):
    vendor = vendors.sample(1).iloc[0]
    gl = gl_accounts[gl_accounts["gl_type"].isin(["Expense", "Revenue"])].sample(1).iloc[0]
    cc = cost_centers.sample(1).iloc[0]

    posting_date = start_date + timedelta(days=random.randint(0, 365))
    invoice_date = posting_date - timedelta(days=random.randint(1, 10))
    amount = round(random.uniform(1000, 60000), 2)

    source_doc_id = f"SRC{i:04d}"
    invoice_id = f"INV{i:05d}"

    invoice_rows.append([
        invoice_id,
        source_doc_id,
        vendor["vendor_id"],
        gl["gl_account"],
        cc["cost_center_id"],
        invoice_date.date().isoformat(),
        posting_date.date().isoformat(),
        amount,
        random.choice(descriptions)
    ])

    payment_delay = random.randint(5, 45)
    payment_date = posting_date + timedelta(days=payment_delay)

    payment_rows.append([
        f"PAY{i:05d}",
        source_doc_id,
        invoice_id,
        payment_date.date().isoformat(),
        round(amount * random.uniform(0.95, 1.00), 2)
    ])

invoices = pd.DataFrame(invoice_rows, columns=[
    "invoice_id", "source_doc_id", "vendor_id", "gl_account", "cost_center_id",
    "invoice_date", "posting_date", "amount_usd", "description"
])

payments = pd.DataFrame(payment_rows, columns=[
    "payment_id", "source_doc_id", "invoice_id", "payment_date", "paid_amount_usd"
])

# Add enterprise-style data quality and control issues
invoices.loc[3, "vendor_id"] = None
invoices.loc[7, "amount_usd"] = -500
invoices.loc[10, "description"] = None

# duplicate-like scenario
invoices.loc[20, ["vendor_id", "amount_usd", "invoice_date", "description"]] = [
    invoices.loc[21, "vendor_id"],
    invoices.loc[21, "amount_usd"],
    invoices.loc[21, "invoice_date"],
    invoices.loc[21, "description"]
]

invoices.to_csv(RAW_DIR / "invoices.csv", index=False)
payments.to_csv(RAW_DIR / "payments.csv", index=False)

print("Transaction data created successfully.")
print("Invoices shape:", invoices.shape)
print("Payments shape:", payments.shape)
display(invoices.head())
display(payments.head())

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, concat_ws, when, trim, upper, datediff, to_date
)

spark = SparkSession.builder.appName("EnterpriseFinanceKnowledgeOpsCopilot").getOrCreate()

base_path = str(RAW_DIR)
processed_path = str(PROCESSED_DIR)
output_path = str(OUTPUT_DIR)

vendors_spark = spark.read.option("header", True).csv(f"{base_path}/vendors.csv")
gl_accounts_spark = spark.read.option("header", True).csv(f"{base_path}/gl_accounts.csv")
cost_centers_spark = spark.read.option("header", True).csv(f"{base_path}/cost_centers.csv")
finance_policies_spark = spark.read.option("header", True).csv(f"{base_path}/finance_policies.csv")
invoices_spark = spark.read.option("header", True).csv(f"{base_path}/invoices.csv")
payments_spark = spark.read.option("header", True).csv(f"{base_path}/payments.csv")

# Standardization
vendors_spark = vendors_spark.withColumn("vendor_name_clean", upper(trim(col("vendor_name"))))
invoices_spark = invoices_spark.withColumn("description_clean", upper(trim(col("description"))))

# Common business key
invoices_spark = invoices_spark.withColumn(
    "common_business_key",
    concat_ws("_", col("vendor_id"), col("cost_center_id"), col("gl_account"))
)

# Integrated finance view
integrated = (
    invoices_spark
    .join(vendors_spark, on="vendor_id", how="left")
    .join(gl_accounts_spark, on="gl_account", how="left")
    .join(cost_centers_spark, on="cost_center_id", how="left")
    .join(
        payments_spark.select("invoice_id", "payment_id", "payment_date", "paid_amount_usd"),
        on="invoice_id",
        how="left"
    )
)

# Date conversion for control checks
integrated = (
    integrated
    .withColumn("posting_date_dt", to_date(col("posting_date")))
    .withColumn("payment_date_dt", to_date(col("payment_date")))
    .withColumn("invoice_date_dt", to_date(col("invoice_date")))
    .withColumn("payment_delay_days", datediff(col("payment_date_dt"), col("posting_date_dt")))
)

# Finance risk / control flags
integrated = (
    integrated
    .withColumn(
        "high_value_flag",
        when(col("amount_usd").cast("double") > 40000, lit("Y")).otherwise(lit("N"))
    )
    .withColumn(
        "vendor_review_flag",
        when(col("vendor_status") == "Review", lit("Y")).otherwise(lit("N"))
    )
    .withColumn(
        "late_payment_flag",
        when(col("payment_delay_days") > 30, lit("Y")).otherwise(lit("N"))
    )
)

# Data quality checks
dq = integrated.select(
    "invoice_id",
    when(col("vendor_id").isNull(), lit("Missing vendor_id"))
    .when(col("amount_usd").cast("double") <= 0, lit("Invalid amount"))
    .when(col("description").isNull(), lit("Missing description"))
    .otherwise(lit(None)).alias("issue_reason")
).filter(col("issue_reason").isNotNull())

# Retrieval-ready finance narrative
retrieval_docs = integrated.withColumn(
    "retrieval_text",
    concat_ws(
        " ",
        lit("Invoice"),
        col("invoice_id"),
        lit("Vendor"),
        col("vendor_name"),
        lit("Vendor Status"),
        col("vendor_status"),
        lit("Cost Center"),
        col("cost_center_name"),
        lit("GL Account"),
        col("gl_account_name"),
        lit("Amount"),
        col("amount_usd"),
        lit("Description"),
        col("description"),
        lit("High Value Flag"),
        col("high_value_flag"),
        lit("Late Payment Flag"),
        col("late_payment_flag")
    )
)

embedding_input = retrieval_docs.select(
    "invoice_id",
    "common_business_key",
    "vendor_id",
    "vendor_name",
    "cost_center_id",
    "cost_center_name",
    "gl_account",
    "gl_account_name",
    "amount_usd",
    "retrieval_text"
)

print("Integrated finance view:")
integrated.show(5, truncate=False)

print("Data quality issues:")
dq.show(truncate=False)

print("Embedding input:")
embedding_input.show(5, truncate=False)

In [ ]:
integrated_pd = integrated.toPandas()
dq_pd = dq.toPandas()
embedding_input_pd = embedding_input.toPandas()
finance_policies_pd = finance_policies_spark.toPandas()

integrated_pd.to_csv(PROCESSED_DIR / "integrated_finance_view.csv", index=False)
dq_pd.to_csv(OUTPUT_DIR / "data_quality_issues.csv", index=False)
embedding_input_pd.to_csv(PROCESSED_DIR / "embedding_input.csv", index=False)
finance_policies_pd.to_csv(PROCESSED_DIR / "finance_policies.csv", index=False)

print("Processed files saved successfully.\n")
print("Saved files:")
for p in sorted(PROCESSED_DIR.glob("*.csv")):
    print("-", p.name)
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", p.name)

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_input_df = pd.read_csv(PROCESSED_DIR / "embedding_input.csv")
finance_policies_df = pd.read_csv(PROCESSED_DIR / "finance_policies.csv")

finance_record_docs = []
for _, row in embedding_input_df.iterrows():
    finance_record_docs.append({
        "doc_id": f"FIN_{row['invoice_id']}",
        "doc_type": "finance_record",
        "source_id": row["invoice_id"],
        "common_business_key": row["common_business_key"],
        "text": row["retrieval_text"],
        "metadata": {
            "invoice_id": row["invoice_id"],
            "vendor_id": row["vendor_id"],
            "vendor_name": row["vendor_name"],
            "cost_center_id": row["cost_center_id"],
            "cost_center_name": row["cost_center_name"],
            "gl_account": row["gl_account"],
            "gl_account_name": row["gl_account_name"],
            "amount_usd": row["amount_usd"]
        }
    })

policy_docs = []
for _, row in finance_policies_df.iterrows():
    policy_docs.append({
        "doc_id": f"POL_{row['policy_id']}",
        "doc_type": "finance_policy",
        "source_id": row["policy_id"],
        "common_business_key": "POLICY",
        "text": f"Policy Name: {row['policy_name']}. Policy Text: {row['policy_text']}. Severity: {row['severity']}.",
        "metadata": {
            "policy_id": row["policy_id"],
            "policy_name": row["policy_name"],
            "severity": row["severity"]
        }
    })

all_docs = finance_record_docs + policy_docs

documents_df = pd.DataFrame([
    {
        "doc_id": d["doc_id"],
        "doc_type": d["doc_type"],
        "source_id": d["source_id"],
        "common_business_key": d["common_business_key"],
        "text": d["text"],
        "metadata_json": json.dumps(d["metadata"])
    }
    for d in all_docs
])

documents_df.to_csv(PROCESSED_DIR / "rag_documents.csv", index=False)

print("RAG documents prepared successfully.")
print("Total documents:", len(documents_df))
display(documents_df.head())

In [ ]:
documents_df = pd.read_csv(PROCESSED_DIR / "rag_documents.csv")

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(embedding_model_name)

document_texts = documents_df["text"].fillna("").tolist()
document_embeddings = embedding_model.encode(
    document_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

np.save(VECTOR_DIR / "document_embeddings.npy", document_embeddings)
documents_df.to_csv(VECTOR_DIR / "vector_documents.csv", index=False)

print("Embeddings generated successfully.")
print("Embedding model:", embedding_model_name)
print("Embeddings shape:", document_embeddings.shape)
display(documents_df.head())

In [ ]:
import faiss

documents_df = pd.read_csv(VECTOR_DIR / "vector_documents.csv")
document_embeddings = np.load(VECTOR_DIR / "document_embeddings.npy")

embedding_matrix = document_embeddings.astype("float32")

faiss_index = faiss.IndexFlatL2(embedding_matrix.shape[1])
faiss_index.add(embedding_matrix)

faiss.write_index(faiss_index, str(VECTOR_DIR / "finance_knowledge.index"))

print("FAISS index created successfully.")
print("Number of documents indexed:", faiss_index.ntotal)
print("Embedding dimension:", embedding_matrix.shape[1])

In [ ]:
documents_df = pd.read_csv(VECTOR_DIR / "vector_documents.csv")
faiss_index = faiss.read_index(str(VECTOR_DIR / "finance_knowledge.index"))

def search_finance_docs(query, top_k=5):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = faiss_index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        row = documents_df.iloc[idx]
        results.append({
            "rank": rank + 1,
            "doc_id": row["doc_id"],
            "doc_type": row["doc_type"],
            "source_id": row["source_id"],
            "common_business_key": row["common_business_key"],
            "text": row["text"],
            "distance": float(distances[0][rank])
        })
    return pd.DataFrame(results)

query = "Show me high value invoices that may need finance approval and policy guidance"
search_results = search_finance_docs(query, top_k=5)

print("Query:", query)
display(search_results)

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

documents_df = pd.read_csv(VECTOR_DIR / "vector_documents.csv")

langchain_docs = []
for _, row in documents_df.iterrows():
    metadata = json.loads(row["metadata_json"]) if pd.notna(row["metadata_json"]) else {}
    metadata["doc_id"] = row["doc_id"]
    metadata["doc_type"] = row["doc_type"]
    metadata["source_id"] = row["source_id"]
    metadata["common_business_key"] = row["common_business_key"]

    langchain_docs.append(
        Document(
            page_content=row["text"],
            metadata=metadata
        )
    )

hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

langchain_vectorstore = FAISS.from_documents(
    documents=langchain_docs,
    embedding=hf_embeddings
)

langchain_vectorstore.save_local(str(VECTOR_DIR / "langchain_faiss_store"))

print("LangChain FAISS vector store created successfully.")
print("Total LangChain documents:", len(langchain_docs))
print("Saved path:", VECTOR_DIR / "langchain_faiss_store")
print("\nSample metadata:")
print(langchain_docs[0].metadata)
print("\nSample text:")
print(langchain_docs[0].page_content[:300])

In [ ]:
retriever = langchain_vectorstore.as_retriever(search_kwargs={"k": 5})

query = "Find high value invoices, vendor review cases, and related finance approval policy"

retrieved_docs = retriever.invoke(query)

print("Query:", query)
print("\nTop retrieved LangChain documents:\n")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"Result {i}")
    print("Metadata:", doc.metadata)
    print("Text:", doc.page_content[:500])
    print("-" * 100)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_structured_finance_response(query, top_k=5):
    retrieved_docs = retriever.invoke(query)

    context = "\n\n".join([doc.page_content for doc in retrieved_docs[:top_k]])

    prompt = f"""
You are an enterprise finance decision copilot.

Use only the given context.
Answer in simple business language.

Return output in this format:
1. Business Summary
2. Risk Level
3. Recommended Action
4. Supporting Evidence

Context:
{context}

Question:
{query}
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {
        "query": query,
        "response": response,
        "retrieved_docs": retrieved_docs[:top_k]
    }

# Test
query = "Which invoices look risky due to high amount, vendor review, or policy issues?"
result = generate_structured_finance_response(query)

print("Query:\n", result["query"])
print("\nStructured AI Response:\n")
print(result["response"])

print("\nTop Retrieved Documents:\n")
for i, doc in enumerate(result["retrieved_docs"], 1):
    print(f"Result {i}")
    print("Metadata:", doc.metadata)
    print("Text:", doc.page_content[:400])
    print("-" * 100)

In [ ]:
documents_df = pd.read_csv(VECTOR_DIR / "vector_documents.csv")

def investigate_finance_query(query, top_k=8):
    retrieved_docs = retriever.invoke(query)

    finance_records = []
    policy_records = []

    for doc in retrieved_docs[:top_k]:
        meta = doc.metadata
        row = {
            "doc_type": meta.get("doc_type"),
            "doc_id": meta.get("doc_id"),
            "source_id": meta.get("source_id"),
            "common_business_key": meta.get("common_business_key"),
            "vendor_name": meta.get("vendor_name"),
            "amount_usd": meta.get("amount_usd"),
            "policy_name": meta.get("policy_name"),
            "severity": meta.get("severity"),
            "text": doc.page_content
        }

        if meta.get("doc_type") == "finance_record":
            finance_records.append(row)
        elif meta.get("doc_type") == "finance_policy":
            policy_records.append(row)

    risky_cases = []
    for row in finance_records:
        text = row["text"]

        reasons = []
        risk_level = "Medium"

        if "Vendor Status Review" in text:
            reasons.append("vendor is under review")
            risk_level = "High"

        if "High Value Flag Y" in text:
            reasons.append("invoice amount is above approval threshold")
            risk_level = "High"

        if "Late Payment Flag Y" in text:
            reasons.append("payment delay is above control threshold")

        if reasons:
            risky_cases.append({
                "invoice_id": row["source_id"],
                "vendor_name": row["vendor_name"],
                "amount_usd": row["amount_usd"],
                "risk_level": risk_level,
                "risk_reason": ", ".join(reasons)
            })

    if risky_cases:
        summary_lines = []
        for case in risky_cases:
            summary_lines.append(
                f"Invoice {case['invoice_id']} from {case['vendor_name']} is {case['risk_level']} risk because {case['risk_reason']}."
            )
        business_summary = " ".join(summary_lines)
    else:
        business_summary = "No strong risky finance records were found in the top retrieved results, but policy review is still recommended."

    applicable_policies = [p["policy_name"] for p in policy_records]
    applicable_policies = list(dict.fromkeys(applicable_policies))

    if applicable_policies:
        supporting_policy_text = ", ".join(applicable_policies)
    else:
        supporting_policy_text = "No direct policy documents were retrieved."

    recommended_action = (
        "Review vendor compliance status, validate approval requirement for high-value invoices, "
        "and check payment control exceptions before downstream approval."
    )

    return {
        "query": query,
        "business_summary": business_summary,
        "recommended_action": recommended_action,
        "supporting_policies": applicable_policies,
        "risky_cases": pd.DataFrame(risky_cases) if risky_cases else pd.DataFrame(),
        "retrieved_docs": retrieved_docs[:top_k]
    }

# Test
query = "Which invoices look risky due to high amount, vendor review, or policy issues?"
result = investigate_finance_query(query)

print("Query:")
print(result["query"])

print("\nBusiness Summary:")
print(result["business_summary"])

print("\nRecommended Action:")
print(result["recommended_action"])

print("\nSupporting Policies:")
print(result["supporting_policies"])

print("\nRisky Cases:")
display(result["risky_cases"])

print("\nRetrieved Documents:")
for i, doc in enumerate(result["retrieved_docs"], 1):
    print(f"Result {i}")
    print("Metadata:", doc.metadata)
    print("Text:", doc.page_content[:350])
    print("-" * 100)

In [ ]:
integrated_finance_df = pd.read_csv(PROCESSED_DIR / "integrated_finance_view.csv")
policies_df = pd.read_csv(PROCESSED_DIR / "finance_policies.csv")

def investigate_invoice(invoice_id):
    record = integrated_finance_df[integrated_finance_df["invoice_id"] == invoice_id].copy()

    if record.empty:
        return {
            "invoice_id": invoice_id,
            "status": "Not Found",
            "business_summary": f"No record found for invoice {invoice_id}.",
            "risk_level": "Unknown",
            "recommended_action": "Check whether the invoice ID is valid.",
            "applicable_policies": []
        }

    row = record.iloc[0]

    reasons = []
    applicable_policies = []
    risk_level = "Low"

    amount_value = float(row["amount_usd"]) if pd.notna(row["amount_usd"]) else 0
    payment_delay = int(row["payment_delay_days"]) if pd.notna(row["payment_delay_days"]) else 0
    vendor_status = str(row["vendor_status"]) if pd.notna(row["vendor_status"]) else ""
    vendor_id = row["vendor_id"] if pd.notna(row["vendor_id"]) else None
    description = row["description"] if pd.notna(row["description"]) else None

    if pd.isna(vendor_id):
        reasons.append("vendor is missing")
        applicable_policies.append("Missing Data Policy")
        risk_level = "High"

    if amount_value <= 0:
        reasons.append("invoice amount is invalid")
        applicable_policies.append("Missing Data Policy")
        risk_level = "High"

    if pd.isna(description):
        reasons.append("description is missing")
        applicable_policies.append("Missing Data Policy")
        risk_level = "High"

    if amount_value > 40000:
        reasons.append("invoice amount is above approval threshold")
        applicable_policies.append("Invoice Policy")
        risk_level = "High"

    if vendor_status == "Review":
        reasons.append("vendor is in review status")
        applicable_policies.append("Vendor Compliance Policy")
        risk_level = "High"

    if payment_delay > 30:
        reasons.append("payment is delayed beyond control threshold")
        applicable_policies.append("Late Payment Policy")
        if risk_level != "High":
            risk_level = "Medium"

    duplicate_check = integrated_finance_df[
        (integrated_finance_df["vendor_id"] == row["vendor_id"]) &
        (integrated_finance_df["amount_usd"] == row["amount_usd"]) &
        (integrated_finance_df["invoice_date"] == row["invoice_date"]) &
        (integrated_finance_df["invoice_id"] != row["invoice_id"])
    ]

    if not duplicate_check.empty:
        reasons.append("possible duplicate invoice pattern found")
        applicable_policies.append("Duplicate Control Policy")
        risk_level = "High"

    applicable_policies = list(dict.fromkeys(applicable_policies))

    if reasons:
        business_summary = (
            f"Invoice {invoice_id} is marked as {risk_level} risk because " +
            ", ".join(reasons) + "."
        )
    else:
        business_summary = (
            f"Invoice {invoice_id} does not currently show major control issues "
            f"based on available finance rules."
        )

    recommended_action = (
        "Validate invoice controls, confirm policy compliance, review vendor and payment status, "
        "and obtain approval if required before downstream release."
    )

    return {
        "invoice_id": invoice_id,
        "status": "Found",
        "business_summary": business_summary,
        "risk_level": risk_level,
        "recommended_action": recommended_action,
        "applicable_policies": applicable_policies,
        "record": record
    }

# Test
invoice_result = investigate_invoice("INV00117")

print("Invoice ID:", invoice_result["invoice_id"])
print("Status:", invoice_result["status"])
print("Risk Level:", invoice_result["risk_level"])
print("\nBusiness Summary:")
print(invoice_result["business_summary"])
print("\nRecommended Action:")
print(invoice_result["recommended_action"])
print("\nApplicable Policies:")
print(invoice_result["applicable_policies"])

if invoice_result["status"] == "Found":
    print("\nInvoice Record:")
    display(invoice_result["record"])

In [ ]:
def investigate_vendor(vendor_id):
    vendor_records = integrated_finance_df[integrated_finance_df["vendor_id"] == vendor_id].copy()

    if vendor_records.empty:
        return {
            "vendor_id": vendor_id,
            "status": "Not Found",
            "business_summary": f"No records found for vendor {vendor_id}.",
            "risk_level": "Unknown",
            "recommended_action": "Check whether the vendor ID is valid.",
            "vendor_summary": pd.DataFrame(),
            "risky_invoices": pd.DataFrame()
        }

    vendor_name = vendor_records["vendor_name"].dropna().iloc[0] if not vendor_records["vendor_name"].dropna().empty else "Unknown Vendor"
    vendor_status = vendor_records["vendor_status"].dropna().iloc[0] if not vendor_records["vendor_status"].dropna().empty else "Unknown"

    vendor_records["amount_usd"] = pd.to_numeric(vendor_records["amount_usd"], errors="coerce")
    vendor_records["payment_delay_days"] = pd.to_numeric(vendor_records["payment_delay_days"], errors="coerce")

    total_invoices = len(vendor_records)
    total_amount = vendor_records["amount_usd"].fillna(0).sum()
    high_value_count = (vendor_records["amount_usd"] > 40000).sum()
    late_payment_count = (vendor_records["payment_delay_days"] > 30).sum()
    missing_data_count = (
        vendor_records["vendor_id"].isna() |
        vendor_records["description"].isna() |
        (vendor_records["amount_usd"] <= 0)
    ).sum()

    duplicate_patterns = vendor_records.groupby(
        ["vendor_id", "amount_usd", "invoice_date"]
    ).size().reset_index(name="dup_count")
    duplicate_count = (duplicate_patterns["dup_count"] > 1).sum()

    risk_level = "Low"
    reasons = []

    if vendor_status == "Review":
        reasons.append("vendor is in review status")
        risk_level = "High"

    if high_value_count > 0:
        reasons.append(f"{high_value_count} invoice(s) are above approval threshold")
        risk_level = "High"

    if late_payment_count > 0:
        reasons.append(f"{late_payment_count} invoice(s) have late payment behavior")
        if risk_level != "High":
            risk_level = "Medium"

    if missing_data_count > 0:
        reasons.append(f"{missing_data_count} invoice(s) have missing or invalid data")
        risk_level = "High"

    if duplicate_count > 0:
        reasons.append(f"{duplicate_count} duplicate-like invoice pattern(s) found")
        risk_level = "High"

    if reasons:
        business_summary = (
            f"Vendor {vendor_id} ({vendor_name}) is marked as {risk_level} risk because " +
            ", ".join(reasons) + "."
        )
    else:
        business_summary = (
            f"Vendor {vendor_id} ({vendor_name}) currently does not show major finance control risks."
        )

    recommended_action = (
        "Review vendor compliance status, validate invoice approval controls, investigate late payments, "
        "and check duplicate or incomplete transactions before releasing further payments."
    )

    risky_invoices = vendor_records[
        (vendor_records["amount_usd"] > 40000) |
        (vendor_records["payment_delay_days"] > 30) |
        (vendor_records["description"].isna()) |
        (vendor_records["amount_usd"] <= 0)
    ][[
        "invoice_id", "vendor_name", "vendor_status", "amount_usd",
        "invoice_date", "payment_date", "payment_delay_days", "description"
    ]].copy()

    vendor_summary = pd.DataFrame([{
        "vendor_id": vendor_id,
        "vendor_name": vendor_name,
        "vendor_status": vendor_status,
        "total_invoices": total_invoices,
        "total_amount_usd": round(float(total_amount), 2),
        "high_value_invoice_count": int(high_value_count),
        "late_payment_count": int(late_payment_count),
        "missing_or_invalid_data_count": int(missing_data_count),
        "duplicate_pattern_count": int(duplicate_count),
        "risk_level": risk_level
    }])

    return {
        "vendor_id": vendor_id,
        "status": "Found",
        "business_summary": business_summary,
        "risk_level": risk_level,
        "recommended_action": recommended_action,
        "vendor_summary": vendor_summary,
        "risky_invoices": risky_invoices
    }

# Test
vendor_result = investigate_vendor("V005")

print("Vendor ID:", vendor_result["vendor_id"])
print("Status:", vendor_result["status"])
print("Risk Level:", vendor_result["risk_level"])
print("\nBusiness Summary:")
print(vendor_result["business_summary"])
print("\nRecommended Action:")
print(vendor_result["recommended_action"])

if vendor_result["status"] == "Found":
    print("\nVendor Summary:")
    display(vendor_result["vendor_summary"])

    print("\nRisky Invoices:")
    display(vendor_result["risky_invoices"])

In [ ]:
airflow_databricks_dag_code = r'''
from airflow import DAG
from airflow.providers.standard.operators.empty import EmptyOperator
from airflow.providers.databricks.operators.databricks import (
    DatabricksCreateJobsOperator,
    DatabricksRunNowOperator,
)
from datetime import datetime

default_args = {
    "owner": "ashok",
    "start_date": datetime(2024, 1, 1),
    "retries": 1
}

job_config = {
    "name": "enterprise_finance_knowledgeops_copilot_job",
    "tasks": [
        {
            "task_key": "run_finance_knowledgeops_notebook",
            "notebook_task": {
                "notebook_path": "/Workspace/Users/your_email@company.com/enterprise_finance_knowledgeops_copilot"
            },
            "new_cluster": {
                "spark_version": "15.4.x-scala2.12",
                "node_type_id": "Standard_DS3_v2",
                "num_workers": 2
            }
        }
    ]
}

with DAG(
    dag_id="enterprise_finance_airflow_to_databricks_trigger",
    default_args=default_args,
    schedule="@daily",
    catchup=False,
    tags=["finance", "airflow", "databricks", "langchain", "huggingface"]
) as dag:

    start_task = EmptyOperator(task_id="start_pipeline")

    create_or_update_job = DatabricksCreateJobsOperator(
        task_id="create_or_update_databricks_job",
        databricks_conn_id="databricks_default",
        json=job_config
    )

    trigger_databricks_job = DatabricksRunNowOperator(
        task_id="trigger_databricks_job",
        databricks_conn_id="databricks_default",
        job_id="{{ ti.xcom_pull(task_ids='create_or_update_databricks_job')['job_id'] }}"
    )

    end_task = EmptyOperator(task_id="end_pipeline")

    start_task >> create_or_update_job >> trigger_databricks_job >> end_task
'''

dag_file_path = DAG_DIR / "enterprise_finance_airflow_to_databricks_trigger.py"
dag_file_path.write_text(airflow_databricks_dag_code, encoding="utf-8")

print("Airflow to Databricks DAG created successfully.")
print("DAG file path:", dag_file_path.resolve())
print("\nDAG preview:\n")
print(dag_file_path.read_text(encoding="utf-8"))

In [ ]:
# Save key investigation outputs
hybrid_result = investigate_finance_query(
    "Which invoices look risky due to high amount, vendor review, or policy issues?"
)
invoice_result = investigate_invoice("INV00117")
vendor_result = investigate_vendor("V005")

# Save risky cases from hybrid investigation
if not hybrid_result["risky_cases"].empty:
    hybrid_result["risky_cases"].to_csv(OUTPUT_DIR / "hybrid_risky_cases.csv", index=False)

# Save invoice drill-down result
if invoice_result["status"] == "Found":
    invoice_result["record"].to_csv(OUTPUT_DIR / f"invoice_{invoice_result['invoice_id']}_investigation.csv", index=False)

# Save vendor summary and risky invoices
if vendor_result["status"] == "Found":
    vendor_result["vendor_summary"].to_csv(OUTPUT_DIR / f"vendor_{vendor_result['vendor_id']}_summary.csv", index=False)
    vendor_result["risky_invoices"].to_csv(OUTPUT_DIR / f"vendor_{vendor_result['vendor_id']}_risky_invoices.csv", index=False)

# Create project summary json
project_summary = {
    "project_name": "Enterprise Finance KnowledgeOps Copilot",
    "core_capabilities": [
        "SAP FICO-style finance integration",
        "Common business key modeling",
        "PySpark data processing",
        "Hugging Face embeddings",
        "FAISS semantic retrieval",
        "LangChain retriever orchestration",
        "Invoice-level investigation",
        "Vendor-level risk investigation",
        "Airflow to Databricks orchestration"
    ],
    "main_outputs": [
        "integrated_finance_view.csv",
        "embedding_input.csv",
        "rag_documents.csv",
        "hybrid_risky_cases.csv",
        f"invoice_{invoice_result['invoice_id']}_investigation.csv" if invoice_result["status"] == "Found" else None,
        f"vendor_{vendor_result['vendor_id']}_summary.csv" if vendor_result["status"] == "Found" else None,
        f"vendor_{vendor_result['vendor_id']}_risky_invoices.csv" if vendor_result["status"] == "Found" else None
    ]
}

project_summary["main_outputs"] = [x for x in project_summary["main_outputs"] if x is not None]

with open(OUTPUT_DIR / "project_summary.json", "w") as f:
    json.dump(project_summary, f, indent=2)

print("Project outputs saved successfully.\n")
print("Output files:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print("-", p.name)

In [ ]:
import zipfile
from pathlib import Path

zip_path = Path("Enterprise-Finance-KnowledgeOps-Copilot-final.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in PROJECT_DIR.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(PROJECT_DIR))

print("Final project zipped successfully.")
print("ZIP path:", zip_path.resolve())

print("\nProject files:")
for p in sorted(PROJECT_DIR.rglob("*")):
    if p.is_file():
        print("-", p)